In [ ]:
import yancc
from yancc.field import Field
from yancc.velocity_grids import MaxwellSpeedGrid, UniformPitchAngleGrid
from yancc.species import LocalMaxwellian, GlobalMaxwellian
from yancc.solve import solve_dke
from yancc.misc import normalize_fluxes_sfincs
from yancc import yancctools


In [ ]:
nx = 5 # resolution in x/ speed coordinate
na = 65 # resolution in  pitch angle coordinate
nt = 17 # resolution in theta / poloidal angle
nz = 33 # resolution in zeta / toroidal angle

speedgrid = MaxwellSpeedGrid(nx)
pitchgrid = UniformPitchAngleGrid(na)

import desc
eq = desc.examples.get("W7-X")
rho = 0.5 # surface label
field = Field.from_desc(eq, rho, nt, nz)
# field = Field.from_vmec(vmec_path, np.sqrt(rho), nt, nz)
# field = Field.from_booz_xform(booz_path, np.sqrt(rho), nt, nz, cutoff=1e-6) # for stellopt or simsopt booz_xform files
# field = Field.from_ipp_bc(bc_path, np.sqrt(rho), nt, nz, cutoff=1e-6) # for IPP booz_xform files

# note all profiles etc should use radial coordinate rho = sqrt(normalized toroidal flux) = r/a

Erho = -2.3e3 # radial electric field Erho = -∂Φ /∂ρ, in Volts

species = [
    GlobalMaxwellian(
        yancc.species.Hydrogen,
        # give full profiles of T, n vs rho in eV and m^-3, 
        temperature=lambda r: 3.0e3 * (1 - r**2),
        density=lambda r: 2e20 * (1 - r**4),
        # then get the value on a single surface
    ).localize(field.rho),
    
    GlobalMaxwellian(
        yancc.species.Electron,
        # give full profiles of T, n vs rho in eV and m^-3, 
        temperature=lambda r: 3.0e3 * (1 - r**2),
        density=lambda r: 2e20 * (1 - r**4),
        # then get the value on a single surface
    ).localize(field.rho)
]

In [ ]:
## 1. Run the physics calculations
erho_grid, flux_diffs, roots = yancctools.find_ambipolar_roots(
    field, pitchgrid, speedgrid, species, erho_min=-10000, erho_max=10000, n_points=11, num_processors=4
)



In [ ]:
# 2. Visualize the results
yancctools.plot_ambipolar_scan(erho_grid, flux_diffs, roots)